# 25 Partitioning Files

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Organize data on storage with <code>partitionBy</code>, see <b>partition pruning</b> skip whole folders, choose good partition columns (and avoid bad ones), understand bucketing conceptually, and know when Databricks recommends <b>liquid clustering</b> instead of partitioning.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Physical layout decides how much data a query reads. A table partitioned by date lets "yesterday's sales" read one folder instead of five years of files. But over-partitioning (by customer ID, say) creates millions of tiny files and makes everything slower. Choosing partition columns is a classic design and interview question. This lesson is <b>interview-critical</b>.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Two different "partitions"</b>
<ul><li><b>In-memory partitions</b> (<code>repartition</code>, <code>spark_partition_id</code>): how rows are split across tasks while a job runs. That topic is in <code>02_spark_internals</code></li><li><b>Storage partitions</b> (<code>partitionBy</code>): how files are organized into <b>folders</b> on disk. <b>This lesson</b></li></ul>
They are related (every in-memory partition writes its own file into every folder it touches), but they are different things.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 How storage partitioning works</b><br>
<code>df.write.partitionBy("country", "sale_date")</code> writes Hive-style folders:<br><code>`</code><br>sales/<br>├── country=IN/<br>│   ├── sale_date=2024-10-01/part-0000.parquet<br>│   └── sale_date=2024-10-02/part-0000.parquet<br>└── country=UK/<br>    └── sale_date=2024-10-01/part-0000.parquet<br><code>`</code>
<ul><li>The partition values live in the <b>folder names</b>, not inside the files</li><li>A filter on a partition column lets Spark skip whole folders: <b>partition pruning</b></li><li>Good partition columns: <b>low cardinality</b>, used in most filters, and giving <b>large partitions</b> (Databricks guidance: at least ~1 GB per partition)</li><li>Bad: high-cardinality columns (IDs, timestamps), which create millions of tiny files</li></ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A payments company stores 3 TB per month of transactions. Partitioning by <code>txn_date</code> (about 100 GB per day) lets the daily reconciliation job read one day. An engineer once added <code>merchant_id</code> as a second partition column: 2 million merchants × 365 days produced over 700 million tiny files, listing the table took longer than the query, and the change was rolled back. Today they use <b>liquid clustering</b> on <code>merchant_id</code> instead.
</div>

## Syntax

```python
df.write.partitionBy("sale_date").mode("overwrite").parquet(path)          # files
df.write.partitionBy("country").mode("overwrite").saveAsTable("cat.sch.t")  # table

spark.read.parquet(path).filter("sale_date = '2024-10-01'")   # partition pruning

# SQL table definitions
spark.sql("CREATE TABLE t (id INT, d DATE) PARTITIONED BY (d)")
spark.sql("CREATE TABLE t (id INT, d DATE) CLUSTER BY (d)")       # liquid clustering (Delta)
```

## Setup: schema, volume and sample data

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates the lesson folder and a sales DataFrame of 200,000 rows over 61 days (October and November 2024) and 4 countries.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The <code>BASE</code> path, then a few sample rows.
</div>

In [0]:
CATALOG, SCHEMA, VOLUME = "workspace", "spark_basics", "raw_files"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/lesson25"
DB = f"{CATALOG}.{SCHEMA}"
print("BASE =", BASE, "| tables in", DB)

In [0]:
import os
from pyspark.sql import functions as F

sales = (
    spark.range(200_000)
    .withColumnRenamed("id", "sale_id")
    .withColumns({
        "sale_date": F.expr("date_add(date'2024-10-01', cast(sale_id % 61 AS INT))"),
        "country": F.element_at(F.array(*[F.lit(c) for c in ["IN", "UK", "AE", "US"]]), ((F.col("sale_id") / 30).cast("int") % 4 + 1).cast("int")),
        "customer_id": (F.col("sale_id") % 5000).cast("int"),
        "amount": F.round((F.col("sale_id") * 7919 % 100000) / 100.0, 2),
    })
)
sales.show(5)

def tree(path, max_lines=12):
    """Print the folder structure under path (first few lines only)."""
    lines = []
    for root, dirs, files in os.walk(path):
        dirs.sort()
        depth = root.replace(path, "").count(os.sep)
        data_files = [f for f in files if not f.startswith((".", "_"))]
        lines.append("  " * depth + os.path.basename(root) + f"/  ({len(data_files)} files)")
    print("\n".join(lines[:max_lines]) + ("\n  ..." if len(lines) > max_lines else ""))

## 1. Write partitioned files and look at the folders

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the data partitioned by <code>country</code>, then by <code>country</code> and <code>sale_date</code>.<br><br>
<b>Why it matters</b><br>Seeing the folders makes partitioning concrete. Notice that the partition columns are <b>removed</b> from the data files and stored only in folder names.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
4 <code>country=...</code> folders. With two levels, each country folder contains 61 <code>sale_date=...</code> folders, which is 244 leaf folders in total.
</div>

In [0]:
sales.write.mode("overwrite").partitionBy("country").parquet(f"{BASE}/by_country")
tree(f"{BASE}/by_country")

sales.write.mode("overwrite").partitionBy("country", "sale_date").parquet(f"{BASE}/by_country_date")
tree(f"{BASE}/by_country_date", max_lines=6)

leaf_folders = sum(1 for root, dirs, _ in os.walk(f"{BASE}/by_country_date") if not dirs)
print("leaf folders:", leaf_folders)

## 2. Partition discovery when reading

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the partitioned folder and shows that <code>country</code> and <code>sale_date</code> come back as columns, rebuilt from the folder names.<br><br>
<b>Why it matters</b><br>Spark discovers partitions from <code>name=value</code> folders automatically and infers their types. Note that the partition columns move to the <b>end</b> of the schema.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A schema ending in <code>country: string</code> and <code>sale_date: date</code>, and 200,000 rows.
</div>

In [0]:
by_cd = spark.read.parquet(f"{BASE}/by_country_date")
by_cd.printSchema()
print("rows:", by_cd.count())

## 3. Partition pruning in the plan

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Filters on partition columns and checks the plan for <code>PartitionFilters</code>, then compares with a filter on a normal column.<br><br>
<b>Why it matters</b><br><code>PartitionFilters</code> means Spark lists and reads <b>only</b> the matching folders. A filter on a non-partition column (<code>amount</code>) can only use file statistics (<code>PushedFilters</code>), so every folder is still opened.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The first plan shows <code>PartitionFilters: [isnotnull(country...), (country = IN), ...sale_date...]</code>. The second shows <code>PartitionFilters: []</code> and the filter under <code>PushedFilters</code> instead.
</div>

In [0]:
print("---- filter on partition columns ----")
by_cd.filter((F.col("country") == "IN") & (F.col("sale_date") == "2024-10-05")).explain()

print("---- filter on a normal column ----")
by_cd.filter(F.col("amount") > 990).explain()

## 4. Measure the difference

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Counts the distinct files that actually supply rows to a query, with and without pruning, using the hidden <code>_metadata.file_path</code> column.<br><br>
<b>Why it matters</b><br>Numbers make it obvious: pruning one day in one country reads 1/244th of the folders.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
All files for the unfiltered read, and only the files of 1 folder for the pruned read.
</div>

In [0]:
def files_read(df):
    return [r[0] for r in df.select(F.col("_metadata.file_path")).distinct().collect()]

all_files = files_read(by_cd)
pruned_files = files_read(by_cd.filter("country = 'IN' AND sale_date = '2024-10-05'"))
print("files without filter:", len(all_files))
print("files after pruning :", len(pruned_files))
print("example:", pruned_files[0].split("by_country_date")[-1])

## 5. Over-partitioning: the small-files disaster

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Partitions by <code>customer_id</code> (5,000 distinct values) and counts folders and files.<br><br>
<b>Why it matters</b><br>High-cardinality partition columns create one folder per value, with a tiny file in each. Listing and opening thousands of small files costs far more than reading the data. A common rule on Databricks: don't partition tables smaller than about 1 TB, and keep each partition at least around 1 GB.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
About 5,000 folders, each holding a file of a few KB, so the average file size is tiny compared with the ~128 MB to 1 GB you want.
</div>

In [0]:
sales.coalesce(4).write.mode("overwrite").partitionBy("customer_id").parquet(f"{BASE}/by_customer")

folders = files = size = 0
for root, dirs, names in os.walk(f"{BASE}/by_customer"):
    data = [n for n in names if not n.startswith((".", "_"))]
    if data:
        folders += 1
        files += len(data)
        size += sum(os.path.getsize(os.path.join(root, n)) for n in data)
print(f"folders: {folders} | files: {files} | average file size: {size / files / 1024:.1f} KB")

## 6. Files per partition: `repartition` by the partition column

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes partitioned data first without, then with <code>repartition</code> on the partition column, and counts files per folder.<br><br>
<b>Why it matters</b><br>Every in-memory partition writes a file into <b>every</b> folder it has rows for. With 8 in-memory partitions and rows for every country in each, you get 8 files per country folder. <code>repartition("country")</code> first sends each country to one task, so each folder gets <b>one</b> file.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
8 files per country folder without the repartition, and 1 file per folder with it.
</div>

In [0]:
def files_per_folder(path):
    return {os.path.basename(r): len([n for n in fs if n.endswith(".parquet")])
            for r, _, fs in os.walk(path) if any(n.endswith(".parquet") for n in fs)}

sales.repartition(8).write.mode("overwrite").partitionBy("country").parquet(f"{BASE}/many_files")
print("without repartition by key:", files_per_folder(f"{BASE}/many_files"))

sales.repartition("country").write.mode("overwrite").partitionBy("country").parquet(f"{BASE}/one_file")
print("with repartition('country'):", files_per_folder(f"{BASE}/one_file"))

## 7. Partitioned tables, and liquid clustering on Databricks

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a partitioned Delta table, shows its partitions, then creates a <b>liquid clustered</b> table with <code>CLUSTER BY</code>.<br><br>
<b>Why it matters</b><br>For Delta tables, Databricks now recommends <b>liquid clustering</b> for most new tables instead of partitioning (and instead of Z-ordering). It clusters data files by the chosen columns without creating folders, adapts to changing query patterns (you can change the keys later), and avoids the small-files problem of high-cardinality partitions. Partitioning is still fine for very large tables with a natural low-cardinality filter such as a date. Clustering and data skipping are covered in depth in <code>03_delta_lake</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
<code>DESCRIBE DETAIL</code> on the partitioned table shows <code>partitionColumns = [country]</code>, and the 4 country values. On the clustered table it shows <code>partitionColumns = []</code> and <code>clusteringColumns = [customer_id]</code>.
</div>

In [0]:
part_table = f"{DB}.lesson25_sales_by_country"
sales.write.mode("overwrite").partitionBy("country").saveAsTable(part_table)
spark.sql(f"DESCRIBE DETAIL {part_table}").select("format", "numFiles", "partitionColumns").show()
spark.table(part_table).select("country").distinct().orderBy("country").show()

In [0]:
clustered_table = f"{DB}.lesson25_sales_clustered"
spark.sql(f"""
    CREATE OR REPLACE TABLE {clustered_table}
    CLUSTER BY (customer_id)
    AS SELECT * FROM {part_table}
""")
display(spark.sql(f"DESCRIBE DETAIL {clustered_table}").select("format", "numFiles", "partitionColumns", "clusteringColumns"))

## 8. Bucketing (concept)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Explains bucketing, which pre-hashes data into a fixed number of files per key.<br><br>
<b>Why it matters</b><br>Bucketing (<code>bucketBy(n, "key")</code>) stores rows with the same key hash in the same bucket file, so two tables bucketed the same way can be joined without a shuffle. It is an interview topic from the Hive/Spark world.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Bucketing works only for Hive-format tables in the legacy metastore. It is <b>not supported</b> for Delta tables or Unity Catalog, so it can't be run in Free Edition. On Databricks, liquid clustering, Z-ordering and AQE cover the same goals. The code below shows the syntax for reference only.
</div>

```python
# Reference only, works on Hive/Parquet tables in a legacy metastore, not on Delta / Unity Catalog
(orders.write.bucketBy(16, "customer_id").sortBy("customer_id")
       .format("parquet").saveAsTable("legacy_db.orders_bucketed"))
```

## Under the hood

```
WRITE  df.write.partitionBy("country")
  each task: for every row, route it to the writer for its country folder
  -> files = (in-memory partitions) x (distinct countries they contain)

READ  spark.read.parquet(path).filter("country = 'IN'")
  1. Driver lists folders: country=IN, country=UK, ...
  2. Partition filter evaluated against FOLDER NAMES -> keep only country=IN
  3. Only files under country=IN are planned as tasks
  (data filters on normal columns are then applied inside those files)
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> `write.partitionBy` runs a job immediately.

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> reading is lazy. Partition pruning happens at planning time, so skipped folders cost nothing.

**Dynamic partition pruning (DPP):** when a partitioned fact table is joined to a filtered dimension (for example `dim_date WHERE year = 2024`), Spark can push the dimension's keys into the fact table scan at runtime and prune partitions even though the filter is on the other table. Look for `dynamicpruningexpression` in plans of such joins.

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A plan where the fact table scan's <code>PartitionFilters</code> contain <code>dynamicpruningexpression(...)</code>.
</div>

In [0]:
# A small date dimension stored as files. DPP needs a real scan on the dimension side,
# not rows typed into the notebook.
(
    spark.createDataFrame([("2024-10-05", "holiday"), ("2024-10-06", "workday")], "day STRING, kind STRING")
    .withColumn("day", F.to_date("day"))
    .write.mode("overwrite").parquet(f"{BASE}/dim_days")
)

dim_days = spark.read.parquet(f"{BASE}/dim_days")
fact = spark.read.parquet(f"{BASE}/by_country_date")
fact.join(dim_days.filter("kind = 'holiday'"), fact.sale_date == dim_days.day).explain()

In [0]:
# Optional cleanup: remove this lesson's tables and files.
for t in ["lesson25_sales_by_country", "lesson25_sales_clustered"]:
    spark.sql(f"DROP TABLE IF EXISTS {DB}.{t}")
dbutils.fs.rm(BASE, True)

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: millions of tiny files, slow listing</b><br>
Partitioned by a high-cardinality column. Partition by a coarser column (date, month) or use liquid clustering.<br><br>
<b>⛔ Problem: many files in every partition folder</b><br>Many in-memory partitions each wrote into every folder. <code>repartition</code> by the partition column before writing.<br><br>
<b>⛔ Problem: query doesn't prune</b><br>The filter isn't on the partition column, or wraps it in something Spark can't evaluate against folder names (for example a UDF). Check <code>PartitionFilters</code> in <code>explain()</code>.<br><br>
<b>⛔ Problem: partition column type changed</b><br>Partition values are inferred from folder names (<code>01</code> becomes <code>1</code>). Set <code>spark.sql.sources.partitionColumnTypeInference.enabled=false</code>, or read through a table.<br><br>
**⛔ Problem: <code>bucketBy</code> fails on Databricks**<br>Bucketing isn't supported for Delta / Unity Catalog tables. Use liquid clustering.<br><br>
<b>⛔ Problem: overwrite deletes all partitions</b><br>Use dynamic partition overwrite or <code>replaceWhere</code> (lesson 24).<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is partitioning in Spark storage, and why use it?</b><br>
<code>partitionBy</code> writes data into folders named after column values, like <code>country=IN/</code>. Queries that filter on those columns read only the matching folders (partition pruning), which reduces IO dramatically.<br><br>

<b>🎤 2. How do you choose a partition column?</b><br>
A low-cardinality column that most queries filter on, typically a date, that produces large partitions (around 1 GB or more each). Never an ID or timestamp. For tables under about 1 TB, Databricks recommends not partitioning at all and using liquid clustering.<br><br>

<b>🎤 3. What is the difference between partition pruning and predicate pushdown?</b><br>
Partition pruning skips whole folders using partition values in the folder names, at planning time. Predicate pushdown passes filters on regular columns to the file reader, which uses file or row-group statistics to skip data inside the selected files.<br><br>

<b>🎤 4. What is the small-files problem and how does partitioning cause it?</b><br>
Each task writes a file per partition folder it touches. With high-cardinality partitions or many tasks, you get huge numbers of tiny files, and listing and opening them dominates query time. Fix it with coarser partitions, repartitioning by the partition column before writing, compaction (<code>OPTIMIZE</code>), or clustering instead of partitioning.<br><br>

<b>🎤 5. What is the difference between <code>repartition</code> and <code>partitionBy</code>?</b><br>
<code>repartition</code> changes how rows are split across tasks in memory during the job. <code>partitionBy</code> controls the folder layout of the output on storage.<br><br>

<b>🎤 6. What is bucketing, and is it used on Databricks?</b><br>
Bucketing hashes rows into a fixed number of files by key, so joins and aggregations on that key can avoid a shuffle. It works for Hive-format tables, but isn't supported for Delta or Unity Catalog. On Databricks you use liquid clustering, Z-ordering and AQE instead.<br><br>

<b>🎤 7. What is liquid clustering?</b><br>
A Delta Lake layout feature (<code>CLUSTER BY</code>) that clusters data files by chosen columns without folder partitions. Keys can be changed without rewriting the table, and it works well for high-cardinality columns. Databricks recommends it for most new tables.<br><br>

<b>🎤 8. What is dynamic partition pruning?</b><br>
When a partitioned table is joined to a filtered dimension, Spark uses the dimension's join keys at runtime to prune partitions of the fact table, even though the filter isn't directly on the fact table.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A 50 GB Delta table is frequently filtered by <code>customer_id</code> (about 2 million distinct values). What should the data engineer do?</b><br>
A. Partition the table by <code>customer_id</code><br>
B. Use liquid clustering on <code>customer_id</code><br>
C. Bucket the table by <code>customer_id</code> into 2 million buckets<br>
D. Partition by <code>customer_id</code> and <code>date</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> Partitioning by a high-cardinality column creates millions of tiny files, and the table is too small to benefit from partitioning. Liquid clustering is the recommended approach.</details><br><br>

<b>Q2. A table is partitioned by <code>event_date</code>. Which query benefits from partition pruning?</b><br>
A. <code>SELECT * FROM t WHERE user_id = 42</code><br>
B. <code>SELECT * FROM t WHERE event_date = '2024-10-01'</code><br>
C. <code>SELECT count(*) FROM t</code><br>
D. <code>SELECT * FROM t ORDER BY event_date</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> Only a filter on the partition column lets Spark skip folders.</details><br><br>

<b>Q3. Which clause creates a Delta table that uses liquid clustering?</b><br>
A. <code>PARTITIONED BY (col)</code><br>
B. <code>CLUSTERED BY (col) INTO 8 BUCKETS</code><br>
C. <code>CLUSTER BY (col)</code><br>
D. <code>ZORDER BY (col)</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b> B is Hive bucketing syntax. <code>ZORDER BY</code> is used with <code>OPTIMIZE</code>, not when creating a table.</details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Write <code>sales</code> partitioned by <b>month</b> (add a <code>sale_month</code> column such as <code>2024-10</code>) with exactly one file per month folder</li><li>Read it back and prove with <code>explain()</code> and the distinct <code>_metadata.file_path</code> count that a filter on <code>sale_month</code> prunes folders</li><li>Show that a filter on <code>sale_date</code> (not a partition column here) does <b>not</b> prune</li><li>In a comment: would you partition this dataset by <code>country</code>, <code>sale_date</code>, <code>sale_month</code> or <code>customer_id</code> if it were 5 TB per year? Why?</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Monthly partitions, one file per folder
monthly = sales.withColumn("sale_month", F.date_format("sale_date", "yyyy-MM"))
monthly.repartition("sale_month").write.mode("overwrite").partitionBy("sale_month").parquet(f"{BASE}/practice/by_month")
print(files_per_folder(f"{BASE}/practice/by_month"))

# 2. Pruning on the partition column
m = spark.read.parquet(f"{BASE}/practice/by_month")
m.filter("sale_month = '2024-10'").explain()
print("files read:", len(files_read(m.filter("sale_month = '2024-10'"))), "of", len(files_read(m)))

# 3. sale_date is a normal column here: PartitionFilters is empty
m.filter("sale_date = '2024-10-05'").explain()

# 4. 5 TB/year is ~14 GB/day: sale_date gives large daily partitions that match the
#    usual "per day" filters. country is too coarse to help much on its own, sale_month
#    is OK but coarser than needed, and customer_id would create millions of tiny files.
#    On Databricks, liquid clustering on (sale_date, country) is also a strong option.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li><code>partitionBy</code> creates <code>col=value</code> folders. Values live in folder names, not in the files</li><li>Partition pruning skips folders when you filter on partition columns (<code>PartitionFilters</code> in the plan)</li><li>Choose low-cardinality, frequently filtered columns with large partitions (~1 GB+). Usually a date</li><li>High-cardinality partitioning causes the small-files problem</li><li><code>repartition(partition_col)</code> before writing gives one file per folder</li><li>Bucketing is not supported on Delta / Unity Catalog. Use liquid clustering (<code>CLUSTER BY</code>)</li><li>Databricks recommends liquid clustering for most new Delta tables, and no partitioning below ~1 TB</li><li>Dynamic partition pruning prunes fact partitions using a filtered dimension in a join</li></ul>
</div>

| Task | Code |
|---|---|
| Partitioned files | `df.write.partitionBy("d").parquet(path)` |
| Partitioned table | `df.write.partitionBy("d").saveAsTable("t")` |
| One file per folder | `df.repartition("d").write.partitionBy("d")...` |
| Check pruning | `explain()` → `PartitionFilters`, distinct `_metadata.file_path` |
| Table layout | `DESCRIBE DETAIL t` → `partitionColumns`, `clusteringColumns` |
| Liquid clustering | `CREATE TABLE t ... CLUSTER BY (col)` |

## Git commit

```
git add 01_spark_basics/25_partitioning_files.ipynb
git commit -m "add 25 partitioning files notebook"
```